# ETL Pipeline - E-commerce Data

**Pipeline:** Raw Data → Extract → Transform → Load

---

## 1. Cấu hình & Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import logging
import warnings
import sqlite3
warnings.filterwarnings('ignore')

# Config
class Config:
    DATA_DIR = Path('data')
    LOG_DIR = Path('logs')
    OUTPUT_DIR = Path('data_clean')
    DB_PATH = Path('data/bigdata.db')  # SQLite database
    
    CSV_FILES = [
        '2019-Oct.csv', '2019-Nov.csv', '2019-Dec.csv',
        '2020-Jan.csv', '2020-Feb.csv'
    ]

Config.LOG_DIR.mkdir(exist_ok=True)
Config.OUTPUT_DIR.mkdir(exist_ok=True)
Config.DB_PATH.parent.mkdir(exist_ok=True)

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# SQLite setup
def get_db_connection():
    """Get SQLite connection"""
    conn = sqlite3.connect(Config.DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_database():
    """Initialize database schema"""
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # Events table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS events (
            event_id INTEGER PRIMARY KEY AUTOINCREMENT,
            event_time TEXT,
            event_type TEXT,
            product_id INTEGER,
            category_id REAL,
            category_code TEXT,
            brand TEXT,
            price REAL,
            user_id INTEGER,
            user_session TEXT,
            source_file TEXT,
            -- Derived features
            date TEXT,
            hour INTEGER,
            day_of_week INTEGER,
            day_name TEXT,
            month INTEGER,
            year INTEGER,
            category_level1 TEXT,
            category_level2 TEXT,
            category_level3 TEXT
        )
    ''')
    
    # Metadata table
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS pipeline_metadata (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            table_name TEXT,
            row_count INTEGER,
            processed_at TEXT,
            status TEXT
        )
    ''')
    
    conn.commit()
    conn.close()
    print(f'✅ Database initialized: {Config.DB_PATH}')

init_database()
print('Setup complete!')

## 2. EXTRACT - Đọc dữ liệu CSV

In [ ]:
def extract_csv(file_path, chunk_size=100000):
    """Đọc CSV theo chunks"""
    chunks = []
    for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
        chunks.append(chunk)
    return pd.concat(chunks, ignore_index=True)

def extract_all_data():
    """Extract tất cả CSV files"""
    all_data = []
    for csv_file in Config.CSV_FILES:
        file_path = Config.DATA_DIR / csv_file
        if file_path.exists():
            print(f'Loading: {csv_file}')
            df = extract_csv(str(file_path))
            df['source_file'] = csv_file
            all_data.append(df)
            print(f'  -> {len(df):,} rows')
    
    combined_df = pd.concat(all_data, ignore_index=True)
    print(f'\nTotal: {len(combined_df):,} rows')
    return combined_df

raw_df = extract_all_data()
raw_df.head()

In [ ]:
# Quick overview
print(f'Shape: {raw_df.shape}')
print(f'Columns: {list(raw_df.columns)}')
print(f'\nMissing values:\n{raw_df.isnull().sum()}')
print(f'\nEvent types:\n{raw_df["event_type"].value_counts()}')

## 3. TRANSFORM - Xử lý dữ liệu

### 3.1 Remove Duplicates

In [ ]:
def remove_duplicates(df):
    before = len(df)
    df = df.drop_duplicates()
    print(f'Remove duplicates: {before:,} -> {len(df):,}')  
    return df

df = remove_duplicates(raw_df)

### 3.2 Handle Missing Values

In [ ]:
def handle_missing_values(df):
    """Xử lý missing values"""
    # category_code, brand: fill với 'unknown'
    df['category_code'] = df['category_code'].fillna('unknown')
    df['brand'] = df['brand'].fillna('unknown')
    
    # user_session: fill với temp id
    df['user_session'] = df['user_session'].fillna(
        pd.Series([f'temp_s_{i}' for i in range(len(df))], index=df.index)
    )
    
    # price: fill với median
    df['price'] = df['price'].fillna(df['price'].median())
    
    # Drop rows missing event_time
    df = df.dropna(subset=['event_time'])
    
    print(f'Missing handled. Remaining: {df.isnull().sum().sum()} nulls')
    return df

df = handle_missing_values(df)

### 3.3 Standardize Categories

In [ ]:
def standardize_categories(df):
    """Chuẩn hóa text fields"""
    df['category_code'] = df['category_code'].str.lower().str.strip()
    df['brand'] = df['brand'].str.lower().str.strip()
    df['category_code'] = df['category_code'].str.replace(r'\s+', ' ', regex=True)
    df['brand'] = df['brand'].str.replace(r'\s+', ' ', regex=True)
    df.loc[df['category_code'] == '', 'category_code'] = 'unknown'
    df.loc[df['brand'] == '', 'brand'] = 'unknown'
    print('Categories standardized')
    return df

df = standardize_categories(df)

### 3.4 Validate Sessions

In [ ]:
def validate_sessions(df):
    before = len(df)
    df = df[df['user_session'].notna()]
    df = df[df['user_session'] != '']
    df = df[df['user_session'].str.len() >= 10]
    print(f'Sessions validated: {before:,} -> {len(df):,}')
    return df

df = validate_sessions(df)

### 3.5 Remove Outliers (IQR)

In [ ]:
def remove_outliers(df):
    before = len(df)
    # Price outliers (1st-99th percentile)
    Q1 = df['price'].quantile(0.01)
    Q3 = df['price'].quantile(0.99)
    df = df[(df['price'] >= Q1) & (df['price'] <= Q3)]
    
    # Validate timestamp
    df['event_time'] = pd.to_datetime(df['event_time'], errors='coerce')
    df = df[df['event_time'].notna()]
    df = df[(df['event_time'] >= '2019-10-01') & (df['event_time'] <= '2020-02-29')]
    
    print(f'Outliers removed: {before:,} -> {len(df):,}')
    return df

df = remove_outliers(df)

### 3.6 Create Features

In [ ]:
def create_features(df):
    """Tạo derived features"""
    df['event_time'] = pd.to_datetime(df['event_time'])
    
    # Date features
    df['date'] = df['event_time'].dt.date
    df['hour'] = df['event_time'].dt.hour
    df['day_of_week'] = df['event_time'].dt.dayofweek
    df['day_name'] = df['event_time'].dt.day_name()
    df['month'] = df['event_time'].dt.month
    df['year'] = df['event_time'].dt.year
    
    # Category hierarchy
    df['category_level1'] = df['category_code'].str.split('.').str[0].fillna('unknown')
    df['category_level2'] = df['category_code'].str.split('.').str.get(1).fillna('unknown')
    df['category_level3'] = df['category_code'].str.split('.').str.get(2).fillna('unknown')
    
    print('Features created')
    return df

df = create_features(df)

## 4. LOAD - Lưu vào SQLite Database

In [ ]:
# Save to SQLite database
def load_to_sqlite(df, table_name='events'):
    """Load DataFrame to SQLite"""
    conn = get_db_connection()
    
    # Convert datetime to string for SQLite
    df_db = df.copy()
    df_db['event_time'] = df_db['event_time'].astype(str)
    df_db['date'] = df_db['date'].astype(str)
    
    # Insert to database
    df_db.to_sql(table_name, conn, if_exists='replace', index=False)
    
    # Update metadata
    cursor = conn.cursor()
    cursor.execute('''
        INSERT INTO pipeline_metadata (table_name, row_count, processed_at, status)
        VALUES (?, ?, ?, ?)
    ''', (table_name, len(df), datetime.now().isoformat(), 'success'))
    
    conn.commit()
    conn.close()
    
    print(f'✅ Loaded {len(df):,} rows to SQLite table: {table_name}')

load_to_sqlite(df)
print(f'\n📁 Database location: {Config.DB_PATH}')

In [ ]:
# Also save CSV as backup
output_path = Config.OUTPUT_DIR / 'events_clean.csv'
df.to_csv(output_path, index=False)
print(f'✅ Backup CSV saved: {output_path}')

In [ ]:
# Verify data in SQLite
conn = get_db_connection()

# Table info
result = conn.execute("SELECT COUNT(*) as count FROM events").fetchone()
print(f'\n📊 Database Verification:')
print(f'   Total rows in events: {result[0]:,}')

# Sample query
sample = pd.read_sql_query('SELECT * FROM events LIMIT 3', conn)
print(f'\n📋 Sample data:')
display(sample)

conn.close()

## 5. Summary

**Pipeline Output:**
- Database: `data/bigdata.db` (SQLite)
- Table: `events` - Cleaned events
- Table: `pipeline_metadata` - Processing history
- Backup CSV: `data_clean/events_clean.csv`